# Carpet WaveToy Thorns

Generate Carpet-style WaveToy thorns and inspect the CCL and source files they provide.

Navigation: [Index](../index.ipynb) | Previous: [ETLegacy Core Infrastructure](etlegacy_core_infrastructure.ipynb) | Next: [Backend Choice Guide](backend_choice_guide.ipynb)


## Generate Carpet WaveToy Thorns
Compiling or running these thorns requires an external Einstein Toolkit checkout. This notebook focuses on generation and inspection.

## Import Thorn Inspection Helpers

These standard-library tools run commands, manage temporary project directories, and clean command output.


In [ ]:
from pathlib import Path
import re, shutil, subprocess, sys, tempfile


def clean_command_output(text):
    cleaned = re.sub(r"\x1b\[[0-?]*[ -/]*[@-~]", "", text or "")
    return cleaned.replace(str(WORKSPACE), "<workspace>")


def run_command(args, cwd, timeout):
    try:
        result = subprocess.run(
            args,
            cwd=cwd,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            check=True,
            timeout=timeout,
        )
    except FileNotFoundError as exc:
        raise RuntimeError(f"Required command is missing: {args[0]}") from exc
    except subprocess.CalledProcessError as exc:
        print(clean_command_output(exc.stdout))
        raise RuntimeError(f"Command failed: {' '.join(map(str, args))}") from exc
    return clean_command_output(result.stdout)


def require_toolchain():
    if shutil.which("make") is None:
        raise RuntimeError(
            "This notebook requires make to build the generated project."
        )
    if not any(shutil.which(name) for name in ["cc", "gcc", "clang"]):
        raise RuntimeError(
            "This notebook requires a C compiler such as cc, gcc, or clang."
        )


## Create a Carpet Thorn Workspace

The workspace keeps generated files separate from the tutorial source tree.


In [ ]:
workspace_manager = tempfile.TemporaryDirectory(
    prefix="nrpy_tutorial_carpet_", dir=Path.cwd()
)
WORKSPACE = Path(workspace_manager.name)
PROJECT_DIR = WORKSPACE / "project" / "et_wavetoy"


## Generate Carpet-Compatible Thorns

This command invokes the same module a learner can run from a terminal and then verifies that the project directory exists.


In [ ]:
command = [sys.executable, "-m", "nrpy.examples.carpet_wavetoy_thorns"]
print("generator command: python -m nrpy.examples.carpet_wavetoy_thorns")
output = run_command(command, WORKSPACE, timeout=300)
for line in output.splitlines():
    if line.strip():
        print(line.rstrip())
if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(PROJECT_DIR)


## Step 4: Inspect the generated inventory

The inventory identifies the generated files relevant to this lesson.

In [ ]:
print("thorn inventory:")
for path in sorted(PROJECT_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(PROJECT_DIR))


## Step 5: Inspect the evolution thorn interface

The interface file declares the evolved variables and inherited Einstein Toolkit capabilities.

In [ ]:
relative_path = "WaveToyNRPy/interface.ccl"
print(f"--- {relative_path} ---")
print((PROJECT_DIR / relative_path).read_text(encoding="utf-8", errors="replace"))


## Step 6: List the evolution thorn schedule entries

The schedule entries show where the RHS and boundary hooks enter the Toolkit time step.

In [ ]:
relative_path = "WaveToyNRPy/schedule.ccl"
print(f"--- {relative_path}: schedule entries ---")
for line in (
    (PROJECT_DIR / relative_path)
    .read_text(encoding="utf-8", errors="replace")
    .splitlines()
):
    if line.startswith("schedule "):
        print(line)


## Step 7: Inspect the initial-data thorn interface

The initial-data interface declares the fields initialized before time evolution begins.

In [ ]:
relative_path = "IDWaveToyNRPy/interface.ccl"
print(f"--- {relative_path} ---")
print((PROJECT_DIR / relative_path).read_text(encoding="utf-8", errors="replace"))


## Step 8: Inspect the diagnostics thorn schedule

The diagnostics schedule shows where output is written during the run.

In [ ]:
relative_path = "WaveToyNRPy/schedule.ccl"
print(f"--- {relative_path}: schedule entries ---")
for line in (
    (PROJECT_DIR / relative_path)
    .read_text(encoding="utf-8", errors="replace")
    .splitlines()
):
    if line.startswith("schedule "):
        print(line)


## Step 9: Inspect One Generated Source File

One RHS source file represents the generated C output path.

In [ ]:
source_path = PROJECT_DIR / "WaveToyNRPy" / "src" / "WaveToyNRPy_rhs_eval.c"
if not source_path.is_file():
    raise FileNotFoundError(source_path)
for line in source_path.read_text(encoding="utf-8", errors="replace").splitlines()[:80]:
    print(line)


The thorn inventory and CCL files show the three pieces of the Carpet WaveToy example: evolution, initial data, and diagnostics. The source snippets are the generated C files named by the thorn build metadata.


## Continue to Backend Choices
- [ETLegacy Core Infrastructure](etlegacy_core_infrastructure.ipynb)
- [Backend Choice Guide](backend_choice_guide.ipynb)
